In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/audio-separation"

folders = [
    "data/speech",
    "data/noise",
    "data/test",
    "model",
    "checkpoints"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Project structure created.")

Project structure created.


In [ ]:
%cd /content/drive/MyDrive/audio-separation

/content/drive/MyDrive/audio-separation


In [ ]:
import os
import tarfile
import urllib.request
import ssl

BASE_DIR = "/content/drive/MyDrive/audio-separation"

SPEECH_DIR = f"{BASE_DIR}/data/speech"

os.makedirs(SPEECH_DIR, exist_ok=True)

url = "https://us.openslr.org/resources/31/train-clean-5.tar.gz"

archive_path = "/content/train-clean-5.tar.gz"

print("Downloading Mini LibriSpeech...")
try:
    unverified_context = ssl._create_unverified_context()
    with urllib.request.urlopen(url, context=unverified_context) as response, \
         open(archive_path, 'wb') as out_file:
        out_file.write(response.read())

    print("Download complete.")
except urllib.error.URLError as e:
    print(f"Download failed due to URLError: {e.reason}")
except Exception as e:
    print(f"An unexpected error occurred during download: {e}")

Download complete.


In [ ]:
with tarfile.open(archive_path, "r:gz") as tar:
    tar.extractall("/content")

print("Extracted.")

/tmp/ipykernel_941/2094820908.py:2: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("/content")


Extracted.


In [ ]:
import glob
import shutil

wav_files = glob.glob(
    "/content/LibriSpeech/train-clean-5/**/*.flac",
    recursive=True
)

print("Found:", len(wav_files), "speech files")

for file in wav_files:
    shutil.copy(
        file,
        SPEECH_DIR
    )

print("Speech copied to:", SPEECH_DIR)

Found: 1519 speech files
Speech copied to: /content/drive/MyDrive/audio-separation/data/speech


In [ ]:
print("Speech files:", len(os.listdir(SPEECH_DIR)))

Speech files: 1519


In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

demand = load_dataset(
    "philgzl/demand",
    split="16k",
    streaming=True
)

print(demand)

README.md:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

IterableDataset({
    features: ['audio', 'name'],
    num_shards: 6
})


In [103]:
import soundfile as sf
import io
import os

NOISE_DIR = f"{BASE_DIR}/data/noise"

os.makedirs(NOISE_DIR, exist_ok=True)

count = 0

for item in demand:
    audio = item["audio"]["array"]
    sr = item["audio"]["sampling_rate"]

    output_path = os.path.join(
        NOISE_DIR,
        f"noise_{count:04d}.wav"
    )

    sf.write(
        output_path,
        audio,
        sr
    )

    count += 1

    if count >= 500:
        break

print("Saved", count, "noise files")

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7919cff01120>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/usr/lib/python3.13/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
  File "/usr/lib/python3.13/multiprocessing/popen_fork.py", line 41, in wait
    if not wait([self.sentinel], timeout):
  File "/usr/lib/python3.13/multiprocessing/connection.py", line 1174, in wait
    ready = selector.select(timeout)
  File "/usr/lib/python3.13/selectors.py", line 398, in select
    fd_event_list = self._selector.poll(timeout)
KeyboardInterrupt: 


KeyboardInterrupt: 

In [ ]:
import random
import torchaudio

speech_file = random.choice(
    os.listdir(SPEECH_DIR)
)

noise_file = random.choice(
    os.listdir(NOISE_DIR)
)

speech_path = os.path.join(
    SPEECH_DIR,
    speech_file
)

noise_path = os.path.join(
    NOISE_DIR,
    noise_file
)

speech, speech_sr = torchaudio.load(
    speech_path
)

noise, noise_sr = torchaudio.load(
    noise_path
)

print("Speech:")
print("  file:", speech_file)
print("  shape:", speech.shape)
print("  sample rate:", speech_sr)

print("\nNoise:")
print("  file:", noise_file)
print("  shape:", noise.shape)
print("  sample rate:", noise_sr)

Speech:
  file: 7367-86737-0098.flac
  shape: torch.Size([1, 76400])
  sample rate: 16000

Noise:
  file: noise_0405.wav
  shape: torch.Size([1, 576006])
  sample rate: 48000


In [ ]:
from IPython.display import Audio, display

display(
    Audio(
        speech_path
    )
)

display(
    Audio(
        noise_path
    )
)

In [ ]:
import os
import random
import torch
import torch.nn.functional as F
import torchaudio

BASE_DIR = "/content/drive/MyDrive/audio-separation"

SPEECH_DIR = f"{BASE_DIR}/data/speech"
NOISE_DIR = f"{BASE_DIR}/data/noise"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints"
TEST_DIR = f"{BASE_DIR}/data/test"

SAMPLE_RATE = 16000

# Keep this small for the Colab T4 (~16 GB VRAM)
SEGMENT_SECONDS = 2
SEGMENT_LENGTH = SAMPLE_RATE * SEGMENT_SECONDS

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

print("Speech files:", len(os.listdir(SPEECH_DIR)))
print("Noise files:", len(os.listdir(NOISE_DIR)))
print("Segment:", SEGMENT_SECONDS, "seconds")


Speech files: 1519
Noise files: 500


In [ ]:
def load_audio(path, target_sr=16000):

    waveform, sr = torchaudio.load(path)

    # Convert stereo → mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # Resample if necessary
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=target_sr
        )
        waveform = resampler(waveform)

    return waveform.squeeze(0)

In [ ]:
speech_file = random.choice(os.listdir(SPEECH_DIR))

speech = load_audio(
    os.path.join(SPEECH_DIR, speech_file)
)

print("Shape:", speech.shape)
print("Duration:", speech.shape[-1] / SAMPLE_RATE)

Shape: torch.Size([235760])
Duration: 14.735


In [ ]:
def crop_or_pad(audio, target_length):

    current_length = audio.shape[-1]

    # Longer → random crop
    if current_length > target_length:

        start = random.randint(
            0,
            current_length - target_length
        )

        audio = audio[
            start:start + target_length
        ]

    # Shorter → zero padding
    elif current_length < target_length:

        audio = F.pad(
            audio,
            (0, target_length - current_length)
        )

    return audio

In [ ]:
speech = crop_or_pad(
    speech,
    SEGMENT_LENGTH
)

print(speech.shape)
print(
    "Duration:",
    speech.shape[-1] / SAMPLE_RATE,
    "seconds"
)

torch.Size([64000])
Duration: 4.0 seconds


In [ ]:
def normalize_audio(audio):

    peak = audio.abs().max()

    if peak > 0:
        audio = audio / peak

    return audio

In [ ]:
def mix_at_snr(speech, noise, snr_db):

    speech_power = torch.mean(speech ** 2)

    noise_power = torch.mean(noise ** 2)

    # Desired noise power for requested SNR
    target_noise_power = (
        speech_power /
        (10 ** (snr_db / 10))
    )

    scale = torch.sqrt(
        target_noise_power /
        (noise_power + 1e-8)
    )

    scaled_noise = noise * scale

    mixture = speech + scaled_noise

    return mixture, speech, scaled_noise

In [ ]:
speech_file = random.choice(
    os.listdir(SPEECH_DIR)
)

noise_file = random.choice(
    os.listdir(NOISE_DIR)
)

speech = load_audio(
    os.path.join(SPEECH_DIR, speech_file)
)

noise = load_audio(
    os.path.join(NOISE_DIR, noise_file)
)

speech = crop_or_pad(
    speech,
    SEGMENT_LENGTH
)

noise = crop_or_pad(
    noise,
    SEGMENT_LENGTH
)

speech = normalize_audio(speech)
noise = normalize_audio(noise)

snr_db = 5

mixture, clean_speech, scaled_noise = mix_at_snr(
    speech,
    noise,
    snr_db
)

print("Mixture:", mixture.shape)
print("Speech:", clean_speech.shape)
print("Noise:", scaled_noise.shape)

Mixture: torch.Size([64000])
Speech: torch.Size([64000])
Noise: torch.Size([64000])


In [ ]:
from IPython.display import Audio, display

print("Original clean speech:")
display(Audio(clean_speech.numpy(), rate=SAMPLE_RATE))

print("Background noise:")
display(Audio(scaled_noise.numpy(), rate=SAMPLE_RATE))

print("Mixed audio:")
display(Audio(mixture.numpy(), rate=SAMPLE_RATE))

Original clean speech:


Background noise:


Mixed audio:


In [ ]:
DEMO_DIR = f"{BASE_DIR}/data/test"

os.makedirs(DEMO_DIR, exist_ok=True)

In [ ]:
torchaudio.save(
    f"{DEMO_DIR}/demo_mixture.wav",
    mixture.unsqueeze(0),
    SAMPLE_RATE
)

torchaudio.save(
    f"{DEMO_DIR}/demo_speech.wav",
    clean_speech.unsqueeze(0),
    SAMPLE_RATE
)

torchaudio.save(
    f"{DEMO_DIR}/demo_noise.wav",
    scaled_noise.unsqueeze(0),
    SAMPLE_RATE
)

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class AudioSeparationDataset(Dataset):

    def __init__(
        self,
        speech_files,
        noise_files,
        sample_rate=16000,
        segment_length=32000
    ):
        self.speech_files = list(speech_files)
        self.noise_files = list(noise_files)

        if not self.speech_files:
            raise ValueError("No speech files found.")

        if not self.noise_files:
            raise ValueError("No noise files found.")

        self.sample_rate = sample_rate
        self.segment_length = segment_length

    def __len__(self):
        return len(self.speech_files)

    def __getitem__(self, index):

        speech_path = self.speech_files[index]
        noise_path = random.choice(self.noise_files)

        speech = load_audio(
            speech_path,
            self.sample_rate
        )

        noise = load_audio(
            noise_path,
            self.sample_rate
        )

        speech = crop_or_pad(
            speech,
            self.segment_length
        )

        noise = crop_or_pad(
            noise,
            self.segment_length
        )

        speech = normalize_audio(speech)
        noise = normalize_audio(noise)

        snr_db = random.uniform(-5, 15)

        mixture, speech, noise = mix_at_snr(
            speech,
            noise,
            snr_db
        )

        return {
            "mixture": mixture,
            "speech": speech,
            "noise": noise
        }


In [ ]:
from sklearn.model_selection import train_test_split

speech_files = sorted([
    os.path.join(SPEECH_DIR, f)
    for f in os.listdir(SPEECH_DIR)
    if f.endswith((".flac", ".wav"))
])

noise_files = sorted([
    os.path.join(NOISE_DIR, f)
    for f in os.listdir(NOISE_DIR)
    if f.endswith((".wav", ".flac"))
])

train_speech_files, val_speech_files = train_test_split(
    speech_files,
    test_size=0.2,
    random_state=42
)

print("Train speech:", len(train_speech_files))
print("Validation speech:", len(val_speech_files))
print("Noise:", len(noise_files))


Speech files: 1519
Noise files: 500


In [ ]:
train_dataset = AudioSeparationDataset(
    speech_files=train_speech_files,
    noise_files=noise_files,
    sample_rate=SAMPLE_RATE,
    segment_length=SEGMENT_LENGTH
)

val_dataset = AudioSeparationDataset(
    speech_files=val_speech_files,
    noise_files=noise_files,
    sample_rate=SAMPLE_RATE,
    segment_length=SEGMENT_LENGTH
)

sample = train_dataset[0]

print("Mixture:", sample["mixture"].shape)
print("Speech:", sample["speech"].shape)
print("Noise:", sample["noise"].shape)


Mixture: torch.Size([64000])
Speech: torch.Size([64000])
Noise: torch.Size([64000])


In [ ]:
difference = (
    sample["mixture"]
    - sample["speech"]
    - sample["noise"]
)

print(
    "Reconstruction error:",
    difference.abs().mean().item()
)


Reconstruction error: 1.0931324734286818e-09


In [ ]:
BATCH_SIZE = 1

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))


In [ ]:
batch = next(iter(train_loader))

print("Mixture:", batch["mixture"].shape)
print("Speech:", batch["speech"].shape)
print("Noise:", batch["noise"].shape)

Mixture: torch.Size([8, 64000])
Speech: torch.Size([8, 64000])
Noise: torch.Size([8, 64000])


In [ ]:
i = 0

display(
    Audio(
        batch["mixture"][i].numpy(),
        rate=SAMPLE_RATE
    )
)

display(
    Audio(
        batch["speech"][i].numpy(),
        rate=SAMPLE_RATE
    )
)

display(
    Audio(
        batch["noise"][i].numpy(),
        rate=SAMPLE_RATE
    )
)

In [ ]:
import torch
import torchaudio

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = torchaudio.models.conv_tasnet_base(
    num_sources=2
)

model = model.to(DEVICE)

print(model)

ConvTasNet(
  (encoder): Conv1d(1, 512, kernel_size=(16,), stride=(8,), padding=(8,), bias=False)
  (mask_generator): MaskGenerator(
    (input_norm): GroupNorm(1, 512, eps=1e-08, affine=True)
    (input_conv): Conv1d(512, 128, kernel_size=(1,), stride=(1,))
    (conv_layers): ModuleList(
      (0): ConvBlock(
        (conv_layers): Sequential(
          (0): Conv1d(128, 512, kernel_size=(1,), stride=(1,))
          (1): PReLU(num_parameters=1)
          (2): GroupNorm(1, 512, eps=1e-08, affine=True)
          (3): Conv1d(512, 512, kernel_size=(3,), stride=(1,), padding=(1,), groups=512)
          (4): PReLU(num_parameters=1)
          (5): GroupNorm(1, 512, eps=1e-08, affine=True)
        )
        (res_out): Conv1d(512, 128, kernel_size=(1,), stride=(1,))
        (skip_out): Conv1d(512, 128, kernel_size=(1,), stride=(1,))
      )
      (1): ConvBlock(
        (conv_layers): Sequential(
          (0): Conv1d(128, 512, kernel_size=(1,), stride=(1,))
          (1): PReLU(num_parameters=

In [ ]:
num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Parameters: {num_params:,}"
)

Parameters: 4,984,881


In [ ]:
batch = next(iter(train_loader))

mixture = batch["mixture"].to(DEVICE)
mixture_input = mixture.unsqueeze(1)

print("Input:", mixture_input.shape)


Input: torch.Size([8, 1, 64000])


In [ ]:
with torch.no_grad():

    output = model(mixture)

print("Output:", output.shape)

Output: torch.Size([8, 2, 64000])


In [ ]:
def si_sdr(
    estimation,
    target,
    eps=1e-8
):
    """
    estimation: [B, T]
    target:     [B, T]
    """

    estimation = estimation - estimation.mean(
        dim=-1,
        keepdim=True
    )

    target = target - target.mean(
        dim=-1,
        keepdim=True
    )

    # Projection of estimation onto target
    projection = (
        torch.sum(
            estimation * target,
            dim=-1,
            keepdim=True
        )
        /
        (
            torch.sum(
                target ** 2,
                dim=-1,
                keepdim=True
            ) + eps
        )
    ) * target

    noise = estimation - projection

    ratio = (
        torch.sum(projection ** 2, dim=-1)
        /
        (
            torch.sum(noise ** 2, dim=-1)
            + eps
        )
    )

    return 10 * torch.log10(
        ratio + eps
    )

In [ ]:
def pit_si_sdr_loss(
    estimates,
    targets
):
    """
    estimates: [B, 2, T]
    targets:   [B, 2, T]
    """

    # Permutation 1:
    # estimate speech → target speech
    # estimate noise  → target noise

    loss_perm_1 = -(
        si_sdr(
            estimates[:, 0],
            targets[:, 0]
        )
        +
        si_sdr(
            estimates[:, 1],
            targets[:, 1]
        )
    )

    # Permutation 2:
    # estimate speech → target noise
    # estimate noise  → target speech

    loss_perm_2 = -(
        si_sdr(
            estimates[:, 0],
            targets[:, 1]
        )
        +
        si_sdr(
            estimates[:, 1],
            targets[:, 0]
        )
    )

    # Pick the better permutation for each sample
    loss = torch.minimum(
        loss_perm_1,
        loss_perm_2
    )

    return loss.mean()

In [ ]:
speech = batch["speech"].to(DEVICE)
noise = batch["noise"].to(DEVICE)

targets = torch.stack(
    [speech, noise],
    dim=1
)

print("Targets:", targets.shape)

Targets: torch.Size([8, 2, 64000])


In [ ]:
loss = pit_si_sdr_loss(
    output,
    targets
)

print("Loss:", loss.item())

Loss: 34.88774108886719


In [ ]:
mixture_sdr = si_sdr(
    mixture.squeeze(1),
    speech
)

print(
    "Average mixture SI-SDR:",
    mixture_sdr.mean().item()
)

Average mixture SI-SDR: 2.8739113807678223


In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

In [102]:
model.train()

batch = next(iter(train_loader))

mixture = batch["mixture"].to(DEVICE)
speech = batch["speech"].to(DEVICE)
noise = batch["noise"].to(DEVICE)

mixture_input = mixture.unsqueeze(1)

targets = torch.stack(
    [speech, noise],
    dim=1
)

optimizer.zero_grad(set_to_none=True)

estimates = model(mixture_input)

loss = pit_si_sdr_loss(
    estimates,
    targets
)

loss.backward()

torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=5.0
)

optimizer.step()

print("One supervised training step completed.")
print("Loss:", loss.item())


One training step completed.
Loss: -43.128971099853516


In [ ]:
# Validate the current supervised model.
# Note: if this model was trained on all speech files before the split,
# this validation is only a diagnostic. For a clean experiment, retrain
# from a fresh model using the train/validation split below.

def validate(model, val_loader, device):
    model.eval()

    total_loss = 0.0
    total_mixture_sdr = 0.0
    total_separated_sdr = 0.0
    total_examples = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            mixture = batch["mixture"].to(
                device, non_blocking=True
            )
            speech = batch["speech"].to(
                device, non_blocking=True
            )
            noise = batch["noise"].to(
                device, non_blocking=True
            )

            estimates = model(mixture.unsqueeze(1))

            targets = torch.stack(
                [speech, noise],
                dim=1
            )

            loss = pit_si_sdr_loss(
                estimates,
                targets
            )

            total_loss += loss.item()

            sdr_0 = si_sdr(
                estimates[:, 0],
                speech
            )

            sdr_1 = si_sdr(
                estimates[:, 1],
                speech
            )

            separated_sdr = torch.maximum(
                sdr_0,
                sdr_1
            )

            mixture_sdr = si_sdr(
                mixture,
                speech
            )

            total_separated_sdr += separated_sdr.sum().item()
            total_mixture_sdr += mixture_sdr.sum().item()
            total_examples += mixture.size(0)

    avg_loss = total_loss / max(len(val_loader), 1)
    avg_mixture_sdr = total_mixture_sdr / total_examples
    avg_separated_sdr = total_separated_sdr / total_examples

    return (
        avg_loss,
        avg_mixture_sdr,
        avg_separated_sdr,
        avg_separated_sdr - avg_mixture_sdr
    )


In [ ]:
val_loss, mixture_sdr, separated_sdr, improvement = validate(
    model,
    val_loader,
    DEVICE
)

print(f"Validation loss: {val_loss:.4f}")
print(f"Mixture SI-SDR: {mixture_sdr:.2f} dB")
print(f"Separated SI-SDR: {separated_sdr:.2f} dB")
print(f"SI-SDR improvement: {improvement:.2f} dB")


In [ ]:
# Clean supervised training run using the train/validation split.
# This is the version to use if you want a leakage-free baseline.

import gc

del model
gc.collect()
torch.cuda.empty_cache()

model = torchaudio.models.conv_tasnet_base(
    num_sources=2
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

EPOCHS = 10
best_val_sdr = float("-inf")
BEST_SUPERVISED_PATH = os.path.join(
    CHECKPOINT_DIR,
    "best_conv_tasnet_supervised.pth"
)

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress:

        mixture = batch["mixture"].to(
            DEVICE,
            non_blocking=True
        )
        speech = batch["speech"].to(
            DEVICE,
            non_blocking=True
        )
        noise = batch["noise"].to(
            DEVICE,
            non_blocking=True
        )

        targets = torch.stack(
            [speech, noise],
            dim=1
        )

        optimizer.zero_grad(set_to_none=True)

        estimates = model(
            mixture.unsqueeze(1)
        )

        loss = pit_si_sdr_loss(
            estimates,
            targets
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.3f}"
        )

    train_loss = running_loss / len(train_loader)

    (
        val_loss,
        mixture_sdr,
        separated_sdr,
        improvement
    ) = validate(
        model,
        val_loader,
        DEVICE
    )

    print(f"\nEpoch {epoch + 1}")
    print(f"Train loss: {train_loss:.4f}")
    print(f"Val loss: {val_loss:.4f}")
    print(f"Mixture SI-SDR: {mixture_sdr:.2f} dB")
    print(f"Separated SI-SDR: {separated_sdr:.2f} dB")
    print(f"SI-SDR improvement: {improvement:.2f} dB")

    if separated_sdr > best_val_sdr:
        best_val_sdr = separated_sdr

        torch.save(
            model.state_dict(),
            BEST_SUPERVISED_PATH
        )

        print(
            f"Saved best model: {best_val_sdr:.2f} dB"
        )


In [ ]:
# Load the best supervised checkpoint.
model.load_state_dict(
    torch.load(
        BEST_SUPERVISED_PATH,
        map_location=DEVICE
    )
)

model.to(DEVICE)
model.eval()

print("Best supervised model loaded.")


In [ ]:
# Free the 2-source model before creating the 4-source MixIT model.
import gc

del model
gc.collect()
torch.cuda.empty_cache()

BATCH_SIZE_MIXIT = 1
MIXIT_SEGMENT_LENGTH = SAMPLE_RATE * 2

self_supervised_model = torchaudio.models.conv_tasnet_base(
    num_sources=4
).to(DEVICE)

print("4-source Conv-TasNet created.")


In [ ]:
class MixITDataset(Dataset):

    def __init__(
        self,
        speech_files,
        noise_files,
        sample_rate=16000,
        segment_length=32000
    ):
        self.speech_files = list(speech_files)
        self.noise_files = list(noise_files)
        self.sample_rate = sample_rate
        self.segment_length = segment_length

    def __len__(self):
        return len(self.speech_files)

    def make_mixture(self):

        speech_path = random.choice(self.speech_files)
        noise_path = random.choice(self.noise_files)

        speech = load_audio(
            speech_path,
            self.sample_rate
        )

        noise = load_audio(
            noise_path,
            self.sample_rate
        )

        speech = crop_or_pad(
            speech,
            self.segment_length
        )

        noise = crop_or_pad(
            noise,
            self.segment_length
        )

        speech = normalize_audio(speech)
        noise = normalize_audio(noise)

        snr_db = random.uniform(-5, 15)

        mixture, _, _ = mix_at_snr(
            speech,
            noise,
            snr_db
        )

        return mixture

    def __getitem__(self, index):

        # Two independent mixtures.
        mixture_a = self.make_mixture()
        mixture_b = self.make_mixture()

        # This is the only input given to the separator.
        mixture_of_mixtures = normalize_audio(
            mixture_a + mixture_b
        )

        return {
            "input": mixture_of_mixtures,
            "mixture_a": mixture_a,
            "mixture_b": mixture_b
        }


In [ ]:
mixit_dataset = MixITDataset(
    speech_files=train_speech_files,
    noise_files=noise_files,
    sample_rate=SAMPLE_RATE,
    segment_length=MIXIT_SEGMENT_LENGTH
)

mixit_loader = DataLoader(
    mixit_dataset,
    batch_size=BATCH_SIZE_MIXIT,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

mixit_sample = mixit_dataset[0]

print("Input:", mixit_sample["input"].shape)
print("Mixture A:", mixit_sample["mixture_a"].shape)
print("Mixture B:", mixit_sample["mixture_b"].shape)


In [ ]:
def mixit_loss(estimates, mixture_a, mixture_b):
    '''
    estimates: [B, 4, T]
    mixture_a: [B, T]
    mixture_b: [B, T]

    We do not compare the four estimated sources with clean
    speech/noise targets. Instead, we find the partition of
    estimated sources that best reconstructs the two observed
    mixtures.
    '''

    # All unique 2-vs-2 partitions of four sources.
    partitions = [
        ((0, 1), (2, 3)),
        ((0, 2), (1, 3)),
        ((0, 3), (1, 2)),
    ]

    best_loss = None

    for group_a, group_b in partitions:

        recon_a = estimates[:, list(group_a), :].sum(dim=1)
        recon_b = estimates[:, list(group_b), :].sum(dim=1)

        # Assignment A -> mixture A, B -> mixture B.
        loss_ab = -(
            si_sdr(recon_a, mixture_a)
            +
            si_sdr(recon_b, mixture_b)
        )

        # Assignment A -> mixture B, B -> mixture A.
        loss_ba = -(
            si_sdr(recon_a, mixture_b)
            +
            si_sdr(recon_b, mixture_a)
        )

        current_loss = torch.minimum(
            loss_ab,
            loss_ba
        ).mean()

        if best_loss is None:
            best_loss = current_loss
        else:
            best_loss = torch.minimum(
                best_loss,
                current_loss
            )

    return best_loss


In [ ]:
# Test the MixIT forward pass and loss before training.
self_supervised_model.eval()

mixit_batch = next(iter(mixit_loader))

mix_input = mixit_batch["input"].to(DEVICE)
mix_a = mixit_batch["mixture_a"].to(DEVICE)
mix_b = mixit_batch["mixture_b"].to(DEVICE)

with torch.no_grad():
    estimates = self_supervised_model(
        mix_input.unsqueeze(1)
    )

print("Input:", mix_input.shape)
print("Estimates:", estimates.shape)

loss = mixit_loss(
    estimates,
    mix_a,
    mix_b
)

print("MixIT loss:", loss.item())


In [ ]:
# One self-supervised MixIT training step.
self_supervised_model.train()

mixit_batch = next(iter(mixit_loader))

mix_input = mixit_batch["input"].to(
    DEVICE,
    non_blocking=True
)

mix_a = mixit_batch["mixture_a"].to(
    DEVICE,
    non_blocking=True
)

mix_b = mixit_batch["mixture_b"].to(
    DEVICE,
    non_blocking=True
)

self_supervised_optimizer = torch.optim.Adam(
    self_supervised_model.parameters(),
    lr=1e-3
)

self_supervised_optimizer.zero_grad(
    set_to_none=True
)

estimates = self_supervised_model(
    mix_input.unsqueeze(1)
)

loss = mixit_loss(
    estimates,
    mix_a,
    mix_b
)

loss.backward()

torch.nn.utils.clip_grad_norm_(
    self_supervised_model.parameters(),
    max_norm=5.0
)

self_supervised_optimizer.step()

print("One MixIT training step completed.")
print("MixIT loss:", loss.item())
